# 💚 PicPay Negócios — Bot de Suporte (versão elaborada)

Versão avançada do bot com:
- **Arquitetura OOP** — classe `PicPayBot` com responsabilidades separadas
- **Streaming** token a token via `generate_content_stream`
- **Interface rica** no terminal via biblioteca `rich`
- **Validação de entrada** com limite de caracteres
- **Retry automático** (3 tentativas com back-off) em falhas de API
- **Exportação de sessão** em `.txt` ao encerrar

---

## Estrutura do código

| Módulo | Responsabilidade |
|---|---|
| `PicPayBot.__init__` | Estado: histórico, log, contador, timestamp |
| `PicPayBot.validate` | Sanitiza entrada do usuário |
| `PicPayBot._prepare_message` | Injeta instrução oculta na 3ª pergunta |
| `PicPayBot._stream_response` | Chama a API com streaming + retry |
| `PicPayBot.ask` | Orquestra o fluxo completo de uma pergunta |
| `PicPayBot.export_session` | Salva transcrição em `.txt` |
| `main()` | Loop interativo com interface `rich` |

## 1. Instalação

In [2]:
# !pip install google-genai rich -q

## 2. Imports e configuração

In [ ]:
import os
import time
import textwrap
from datetime import datetime
from typing import Iterator

from rich.console import Console
from rich.panel import Panel
from rich.progress import Progress, SpinnerColumn, TextColumn
from rich.rule import Rule
from rich.table import Table
from rich.text import Text
from rich.live import Live
from rich.padding import Padding

from google import genai
from google.genai import types

API_KEY     = os.environ.get("GEMINI_API_KEY", "SUA_CHAVE_AQUI")
MODEL       = "gemini-2.5-flash"
MAX_Q       = 3
MAX_RETRIES = 3
MAX_INPUT   = 400

console = Console()

## 3. System Instruction (knowledge base fechada)

In [14]:
SYSTEM_INSTRUCTION = """
Você é a Pi, assistente virtual de suporte do Portal PicPay Negócios para Lojistas.

## Personalidade
- Tom moderno, descontraído e objetivo — no estilo da marca PicPay
- Linguagem clara, sem juridiquês ou termos bancários complicados
- NUNCA inventa procedimentos — responde SOMENTE com base no conhecimento abaixo
- Se a pergunta estiver fora do escopo do Portal PicPay Negócios, informe educadamente
- Respostas sempre em português brasileiro

## Tarefa
- Responder EXATAMENTE 3 perguntas nesta sessão
- Na 3ª resposta: após responder normalmente, adicionar a seção
  '📋 RESUMO DO ATENDIMENTO' com um parágrafo coeso resumindo as três
  trocas desta conversa, e encerrar com uma despedida cordial

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
KNOWLEDGE BASE — PICPAY NEGÓCIOS (PORTAL INTERNO PARA LOJISTAS)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

### Acesso ao portal
- URL: https://lojista.picpay.com
- Login com e-mail e senha cadastrados no credenciamento
- App: \"PicPay Negócios\" (iOS e Android)
- Perda de acesso: \"Esqueci a senha\" na tela de login ou chat no portal

### Taxas de transação (MDR)
- PicPay Carteira (saldo PicPay): 0,99%
- Crédito à vista: 2,99%  |  Crédito 2x–6x: 3,49%  |  Crédito 7x–12x: 3,99%
- Débito: 1,49%           |  Pix: 0,99%
⚠️ Volume mensal > R$ 50.000: solicitar renegociação em \"Meu Plano\" > \"Solicitar Revisão de Taxa\" — análise em 5 dias úteis

### Prazo de recebimento (liquidação)
- PicPay Carteira, Pix, Débito: D+1
- Crédito à vista: D+30 (ou D+2 com antecipação automática)
- Crédito parcelado: cada parcela em D+30 da data da transação
Ativar antecipação: \"Financeiro\" > \"Antecipação\" > toggle \"Antecipação Automática\"
Taxa: 1,80% ao mês. Confirmação em até 1h útil.

### Contestação de chargeback
Chargeback = portador contesta compra junto à operadora.
Passo a passo:
  1. \"Financeiro\" > \"Chargebacks\" > clique em \"Aguardando Documentação\"
  2. Anexe: nota fiscal, comprovante de entrega, print do pedido
  3. \"Enviar Contestação\" — prazo: 10 dias corridos após notificação
  4. Resultado: até 30 dias úteis por e-mail
⚠️ Taxa de chargeback > 1% das transações mensais → monitoramento automático

### Bloqueio e limite
Causas: chargeback > 1%, suspeita de fraude, documentação vencida, descumprimento dos termos
Regularizar: \"Minha Conta\" > \"Status do Cadastro\" → verificar pendências
Contestar bloqueio: \"Suporte\" > \"Abrir Chamado\" > \"Bloqueio de Conta\" — análise em 3 dias úteis

### Saque e transferência
- Para conta PJ do titular: grátis, D+1
- Para outras contas: R$ 2,50 por saque, D+1
- Limite diário: R$ 20.000
- Ampliar limite: \"Financeiro\" > \"Limites\" > \"Solicitar Aumento\" + extrato 3 meses

### Relatórios e comprovantes
- \"Relatórios\" > período + formato (PDF/CSV) > \"Gerar Relatório\"
- Tipos: vendas, chargebacks, antecipações, repasses
- Comprovante avulso: clique na transação > \"Baixar Comprovante\"
- NF-e: NÃO emitida pelo PicPay — emitir pelo próprio sistema fiscal

### Cancelamento de transação
- Até 24h após captura: \"Vendas\" > transação > \"Cancelar Transação\" + motivo
- Após 24h: \"Suporte\" > \"Estorno Manual\"
- Estorno ao comprador: até 7 dias úteis

### PicPay Negócios Plus (volume > R$ 10.000/mês)
- Pix 0,75% | Antecipação 1,50% a.m. | Suporte prioritário | Gerente dedicado
- Aderir: \"Meu Plano\" > \"Conhecer PicPay Negócios Plus\"

### Suporte direto
- Chat portal/app: 24/7
- Telefone: 4004-5555 (capitais) | 0800 722 5555 — seg–sex 8h–20h, sáb 9h–15h
- E-mail financeiro: disputas.lojista@picpay.com
"""

## 4. Classe `PicPayBot`

Encapsula todo o estado e comportamento do bot.

In [15]:
class PicPayBot:
    """Bot de suporte PicPay com streaming, retry e exportação de sessão."""

    def __init__(self):
        self.client     = genai.Client(api_key=API_KEY)
        self.history: list[types.Content] = []
        self.log: list[dict] = []
        self.q_count    = 0
        self.started_at = datetime.now()

    # ── Validação ──────────────────────────────────────────────────────────────
    @staticmethod
    def validate(text: str) -> tuple[bool, str]:
        text = text.strip()
        if not text:
            return False, "Digite sua pergunta antes de enviar."
        if len(text) > MAX_INPUT:
            return False, f"Pergunta muito longa ({len(text)} chars). Máximo: {MAX_INPUT}."
        return True, text

    # ── Injeção de instrução oculta na última pergunta ─────────────────────────
    def _prepare_message(self, user_text: str) -> str:
        if self.q_count + 1 >= MAX_Q:
            return (
                user_text
                + "\n\n[INSTRUÇÃO INTERNA — NÃO EXIBIR: "
                "Esta é a 3ª e última pergunta. Após responder normalmente, "
                "adicione a seção '📋 RESUMO DO ATENDIMENTO' resumindo as três "
                "trocas desta sessão em um parágrafo coeso. Encerre com despedida cordial.]"
            )
        return user_text

    # ── Streaming com retry e exponential back-off ─────────────────────────────
    def _stream_response(self, prepared_text: str) -> Iterator[str]:
        self.history.append(
            types.Content(role="user", parts=[types.Part(text=prepared_text)])
        )
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                stream = self.client.models.generate_content_stream(
                    model=MODEL,
                    config=types.GenerateContentConfig(
                        system_instruction=SYSTEM_INSTRUCTION,
                    ),
                    contents=self.history,
                )
                full_text = ""
                for chunk in stream:
                    piece = chunk.text or ""
                    full_text += piece
                    yield piece
                self.history.append(
                    types.Content(role="model", parts=[types.Part(text=full_text)])
                )
                return
            except Exception as exc:
                if attempt < MAX_RETRIES:
                    console.print(f"  [yellow]⚠ Tentativa {attempt} falhou. Aguardando {attempt * 2}s...[/yellow]")
                    time.sleep(attempt * 2)
                else:
                    self.history.pop()
                    raise RuntimeError(f"API indisponível após {MAX_RETRIES} tentativas: {exc}") from exc

    # ── Orquestra uma pergunta completa ────────────────────────────────────────
    def ask(self, user_input: str) -> bool:
        """Retorna True se esta foi a última pergunta."""
        ok, user_input = self.validate(user_input)
        if not ok:
            console.print(f"  [red]{user_input}[/red]\n")
            return False

        self.q_count += 1
        is_last  = self.q_count >= MAX_Q
        prepared = self._prepare_message(user_input)

        console.print()
        console.print(Text(f"  [{self.q_count}/{MAX_Q}] Você", style="bold green"))
        console.print(Panel(user_input, border_style="green", padding=(0, 2)))
        console.print()
        console.print(Text("  Pi", style="bold magenta"))

        response_text = ""
        with Live(console=console, refresh_per_second=20) as live:
            buffer = ""
            for token in self._stream_response(prepared):
                buffer += token
                response_text = buffer
                live.update(Panel(buffer, border_style="magenta", padding=(0, 2)))

        self.log.append({
            "q": self.q_count,
            "timestamp": datetime.now().isoformat(),
            "user": user_input,
            "bot": response_text,
        })
        return is_last

    # ── Exporta transcrição ────────────────────────────────────────────────────
    def export_session(self) -> str:
        now  = datetime.now().strftime("%Y%m%d_%H%M%S")
        path = f"picpay_sessao_{now}.txt"
        lines = [
            "=" * 62,
            "  PicPay Negócios — Transcrição de Atendimento",
            f"  Início: {self.started_at.strftime('%d/%m/%Y %H:%M:%S')}",
            f"  Fim:    {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}",
            "=" * 62, "",
        ]
        for entry in self.log:
            lines += [
                f"[{entry['q']}/{MAX_Q}] Você ({entry['timestamp']}):",
                textwrap.fill(entry["user"], width=60, initial_indent="  "),
                "",
                "Pi:",
                textwrap.fill(entry["bot"], width=60, initial_indent="  "),
                "", "-" * 62, "",
            ]
        with open(path, "w", encoding="utf-8") as f:
            f.write("\n".join(lines))
        return path

    # ── Tabela de resumo ───────────────────────────────────────────────────────
    def print_session_table(self):
        table = Table(title="Resumo da sessão", header_style="bold magenta", border_style="magenta")
        table.add_column("#",        width=3, justify="center")
        table.add_column("Pergunta", width=38)
        table.add_column("Horário",  width=8, justify="center")
        for e in self.log:
            short = (e["user"][:35] + "…") if len(e["user"]) > 35 else e["user"]
            table.add_row(str(e["q"]), short, e["timestamp"][11:16])
        console.print(Padding(table, (1, 2)))

## 5. Funções de UI

In [16]:
def print_welcome():
    console.print()
    console.print(Panel(
        "[bold green]💚  PicPay Negócios  |  Suporte ao Lojista[/bold green]\n\n"
        "Olá! Sou a [bold magenta]Pi[/bold magenta], assistente do Portal PicPay.\n"
        f"Esta sessão aceita [bold]{MAX_Q} perguntas[/bold] sobre o portal interno.\n\n"
        "[dim]Tópicos: taxas MDR, chargebacks, antecipação, bloqueios, saques...[/dim]",
        border_style="green", padding=(1, 4),
    ))
    sug = Table(show_header=False, box=None, padding=(0, 2))
    sug.add_column(style="dim")
    sug.add_column(style="italic")
    sug.add_row("💡", "Como cancelo um pedido já aceito?")
    sug.add_row("💡", "Recebi um chargeback — o que faço?")
    sug.add_row("💡", "Como ativo a antecipação automática?")
    console.print(Padding(sug, (0, 2)))
    console.print()


def print_farewell(bot: PicPayBot):
    console.print()
    console.print(Rule(style="magenta"))
    bot.print_session_table()
    path = bot.export_session()
    console.print(Panel(
        f"[green]✔[/green] Transcrição salva em [bold]{path}[/bold]\n\n"
        "[dim]Boas vendas! 💚[/dim]",
        border_style="green", padding=(0, 2),
    ))
    console.print()

## 6. Execute o bot aqui! 💚

| # | Pergunta sugerida |
|---|---|
| 1 | `Como cancelo um pedido já aceito?` |
| 2 | `Recebi um chargeback, o que faço?` |
| 3 | `Como ativo a antecipação automática?` |

In [17]:
bot = PicPayBot()
print_welcome()

while bot.q_count < MAX_Q:
    remaining   = MAX_Q - bot.q_count
    prompt_text = f"[green]Pergunta {bot.q_count + 1}/{MAX_Q}[/green] [dim]({remaining} restante{'s' if remaining != 1 else ''})[/dim] › "
    try:
        user_input = console.input(prompt_text)
    except (KeyboardInterrupt, EOFError):
        console.print("\n[dim]Atendimento interrompido.[/dim]")
        break
    try:
        if bot.ask(user_input):
            break
    except RuntimeError as err:
        console.print(f"\n[red]Erro: {err}[/red]\n")

print_farewell(bot)

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│    💚  PicPay Negócios  |  Suporte ao Lojista                                                                   │
│                                                                                                                 │
│    Olá! Sou a Pi, assistente do Portal PicPay.                                                                  │
│    Esta sessão aceita 3 perguntas sobre o portal interno.                                                       │
│                                                                                                                 │
│    Tópicos: taxas MDR, chargebacks, antecipação, bloqueios, saques...                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

  💡    Como cancelo um pedido já aceito?                                                                        
    💡    Recebi um chargeback — o que faço?                                                                       
    💡    Como ativo a antecipação automática?  

Pergunta 1/3 (3 restantes) ›

  [1/3] Você

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│  Como cancelo um pedido já aceito?                                                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│  Como ativo a antecipação automática?                                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

  Pi

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────

                     Resumo da sessão                                                                            
  ┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┓                                                      
  ┃  #  ┃ Pergunta                               ┃ Horário  ┃                                                      
  ┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━┩                                                      
  │  1  │ Como cancelo um pedido já aceito?      │  14:30   │                                                      
  │  2  │ Recebi um chargeback, o que faço?      │  14:30   │                                                      
  │  3  │ Como ativo a antecipação automática…   │  14:31   │                                                      
  └─────┴────────────────────────────────────────┴──────────┘

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│  ✔ Transcrição salva em picpay_sessao_20260325_143107.txt                                                       │
│                                                                                                                 │
│  Boas vendas! 💚                                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯